In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns



## Load the dataset

In [3]:
# Load the preprocessed DataFrame
df = pd.read_pickle("../data/interim/df_dtypes_fixed.pkl")

## Define variables

In [4]:
X = df.drop(columns = ['is_fraud', 'fraud_type', 'transaction_id'])

We drop the **`fraud_type`** column, as it is not suitable for predicting fraudulent transactions and may introduce unnecessary complexity.
We also drop the **`transaction_id`** column, as it serves only as an identifier and does not provide predictive value.

In [5]:
y = df['is_fraud']

## Train/Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

We use **stratification** because the target class is heavily imbalanced, and we want to preserve the distribution of the fraud class.

## Handling missing values

In [29]:
X_train.isna().sum()

timestamp                                  2
sender_account                             0
receiver_account                           0
amount                                     0
transaction_type                           0
merchant_category                          0
location                                   0
device_used                                0
time_since_last_transaction            28432
spending_deviation_score                   0
velocity_score                             0
geo_anomaly_score                          0
payment_channel                            0
ip_address                                 0
device_hash                                0
time_since_last_transaction_missing        0
hour_of_day                                2
weekday                                    2
dtype: int64

We drop the two rows with missing **`timestamp`** values, as they represent a negligible portion of the dataset.

We will use **User-Level Median Imputation** to handle missing values in the **`time_since_last_transaction`** column. This strategy preserves each user's typical behavior and minimizes the impact on fraud detection signals.  

The median will be computed **only on the training dataset**, and the same transformation will be applied to the test dataset.

TypeError: DataFrame.dropna() got an unexpected keyword argument 'columns'

In [7]:
# Compute per-sender median on training data
user_medians = X_train.groupby('sender_account')['time_since_last_transaction'].median()

# Map medians to the training data
X_train['time_since_last_transaction'] = X_train['time_since_last_transaction'].fillna(
    X_train['sender_account'].map(user_medians)
)


In [8]:
# Count remaining missing values
remaining_missing = X_train['time_since_last_transaction'].isna().sum()

# Calculate percentage
total_rows = len(X_train)
missing_percent = (remaining_missing / total_rows) * 100

print(f"Remaining missing values: {remaining_missing}")
print(f"Percentage of missing values: {missing_percent:.2f}%")

Remaining missing values: 28432
Percentage of missing values: 0.71%


Approximately **0.71% of the rows** belong to users for whom all values in the **`time_since_last_transaction`** column are missing. These missing values are likely **not due to data errors**; instead, they correspond to new users who appear only once in the dataset. In this case, leaving the values as NaN is appropriate.  

We will also adjust the missing-value flag to capture only users with entirely missing values in this column.  

For such users, leaving **`time_since_last_transaction`** as NaN preserves behavioral information instead of averaging it away. This missingness is **informative for fraud detection**, signaling to the model that the account is new or low-activity, which may correlate with certain types of fraud.

In [9]:
# adjust the missing-value flag to capture only users with entirely missing history, not just individual transactions
X_train['time_since_last_transaction_missing'] = X_train['time_since_last_transaction'].isnull().astype(int)

Next, we apply the same logic to the test dataset, completing the missing-value imputation process.

In [10]:
# Map training medians to test data
X_test['time_since_last_transaction'] = X_test['time_since_last_transaction'].fillna(
    X_test['sender_account'].map(user_medians)
)

# Add missingness flag for test data
X_test['time_since_last_transaction_missing'] = X_test['time_since_last_transaction'].isna().astype(int)

## Fixing problemetic values

Next, we examine the features **`amount`** and **`time_since_last_transaction`** for invalid negative values, out-of-range values, extreme outliers, and unusual distributions, while excluding other numeric features as they were previously identified as well-engineered in the initial notebook.
In addition, we examine the **`timestamp`** column to determine the time range covered by the observations.

In [11]:
stats_table = X_train[['amount', 'time_since_last_transaction', 'timestamp']].describe(percentiles=[.01, .25, .5, .75, .9, .99])
stats_table

,amount,time_since_last_transaction,timestamp
count,4.000000e+06,3.971568e+06,3999998
mean,3.586043e+02,1.186623e+00,2023-07-02 23:30:32.073420
min,1.000000e-02,-8.777814e+03,2023-01-01 00:09:26.241974
1%,1.000000e-02,-7.415783e+03,2023-01-05 03:05:17.076935
25%,2.654000e+01,-2.207119e+03,2023-04-02 17:57:40.075809
50%,1.383300e+02,1.814732e+00,2023-07-03 00:24:31.386949
75%,5.032500e+02,2.210682e+03,2023-10-02 03:57:41.445373
90%,1.125950e+03,4.523218e+03,2023-11-25 22:07:35.249739
99%,1.874110e+03,7.414888e+03,2023-12-28 18:58:27.601108
max,3.520570e+03,8.757758e+03,2024-01-01 22:58:30.131850


In [12]:
X_train["spending_deviation_score"].describe()


count    4.000000e+06
mean    -3.683350e-04
std      1.000915e+00
min     -5.260000e+00
25%     -6.800000e-01
50%      0.000000e+00
75%      6.700000e-01
max      5.020000e+00
Name: spending_deviation_score, dtype: float64

### amount
- Strong right skew (median ≈ 138 vs mean ≈ 359)
- Large but realistic outliers (max ≈ 3520)
- No obvious errors such as negative values → we keep it, but are going to transform it later (log)

### timestamp
- Covers full year (Jan 2023 → Jan 2024)
- Even distribution → suitable for time-based feature extraction







### time_since_last_transaction

An analysis of the descriptive statistics for **`time_since_last_transaction`** indicates that this feature is **standardized (scaled and centered)** rather than representing a raw temporal measurement or resulting from a sorting error.  

**Key evidence for standardization:**
- **Near-zero central tendency:** The mean (≈ 1.18) and median (≈ 1.81) are negligible compared to the standard deviation (≈ 3,353), indicating mean-centering.  
- **Symmetry:** The percentiles are nearly mirrored around zero (e.g., 25th percentile ≈ -2,207 vs. 75th percentile ≈ 2,210), a typical characteristic of standardized features.  
- **Exclusion of sorting errors:** While unsorted data can produce negative values, the consistent and balanced distribution strongly suggests a deliberate transformation (e.g., standard scaling).  

**Interpretation of values:**
- Negative values indicate transaction intervals shorter than the average.  
- Positive values indicate intervals longer than the average.  

Thus, negative values do not represent invalid or corrupted data, but rather reflect relative deviations from the mean.

## Feature Construction
### Time-Based Features

In [19]:
X_train['hour_of_day'] = X_train['timestamp'].dt.hour

In [23]:
X_train['weekday'] = X_train['timestamp'].dt.weekday # Monday=0, Sunday=6

In [28]:
X_train["timestamp"][X_train["weekday"].isna()]

2833396   NaT
3214871   NaT
Name: timestamp, dtype: datetime64[us]

In [17]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 4000000 entries, 1867199 to 2779601
Data columns (total 16 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   timestamp                            datetime64[us]
 1   sender_account                       str           
 2   receiver_account                     str           
 3   amount                               float64       
 4   transaction_type                     str           
 5   merchant_category                    str           
 6   location                             str           
 7   device_used                          str           
 8   time_since_last_transaction          float64       
 9   spending_deviation_score             float64       
 10  velocity_score                       int64         
 11  geo_anomaly_score                    float64       
 12  payment_channel                      str           
 13  ip_address                           

1867199   2023-05-12 07:28:52.763549
2003209   2023-05-12 02:57:29.558504
1024599   2023-05-18 21:34:18.276721
4901380   2023-03-21 21:41:08.695810
1634088   2023-02-25 18:21:51.062387
Name: timestamp, dtype: datetime64[us]